# Ingestion with heading chunks and image documents

Notebook 05 packed each page into chunks of up to 3,000 characters and attached the page's images to every chunk.
This notebook tries another approach on the same seven pages. It reads notebook 04's cached parse,
so no LlamaParse credits are spent.

| Step | What it does |
|---|---|
| 1. Load | Read the cached LlamaParse result: one markdown string per page, and the cropped images |
| 2. Chunk | Split each page on its own at its headings, with LangChain's `MarkdownHeaderTextSplitter`. A section shorter than `MIN_CHUNK_CHARACTERS` joins the next one. |
| 3. Images | Each cropped figure or chart becomes its own document. Table crops, and crops lying inside a larger crop, are skipped. |
| 4. Describe | DeepSeek Flash describes each image, with its page's text as context. Text chunks, tables included, are not described. |
| 5. Store | Text chunks are embedded as they are, image documents by their description, with the image path in the metadata |
| 6. Search | The questions from notebook 05, to compare |

What changes from notebook 05:

- **No maximum chunk size.** Each page is split on its own, so a chunk can never be longer than its page.
- **Tables are embedded as raw HTML**, not by a description. This is the main thing to check.
- **An image reaches the answering model only when its own description is retrieved**, instead of travelling with every chunk of its page.

Run notebook 04 first if `notebooks/llamaparse_output/` is missing.

In [1]:
import base64
import json
import os
import sys
import time
from collections import Counter
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_text_splitters import MarkdownHeaderTextSplitter

PROJECT_ROOT = Path.cwd().parent  # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from multimodal_rag import config
from multimodal_rag.store import NemotronEmbeddings

LLAMAPARSE_OUTPUT = PROJECT_ROOT / "notebooks" / "llamaparse_output"
PARSE_RESULT_PATH = LLAMAPARSE_OUTPUT / "sample_result.json"            # written by notebook 04
IMAGE_DIRECTORY = LLAMAPARSE_OUTPUT / "images"                          # written by notebook 04
DESCRIPTIONS_PATH = LLAMAPARSE_OUTPUT / "image_descriptions.json"       # cache, so a re-run skips the model calls
TEST_STORE_DIRECTORY = LLAMAPARSE_OUTPUT / "chroma_test_06"

# Image description model, on DeepSeek's own API. Its key is DEEPSEEK_API_KEY in .env.
DESCRIPTION_MODEL = "deepseek-flash"  # DeepSeek-V4.1-Flash, reads text and images
DESCRIPTION_BASE_URL = "https://api.deepseek.com"

## 1. Load the parse result

In [2]:
parse_result = json.loads(PARSE_RESULT_PATH.read_text(encoding="utf-8"))

markdown_by_page = {}
for page in parse_result["markdown"]["pages"]:
    markdown_by_page[page["page_number"]] = page["markdown"]

for page_number, markdown in markdown_by_page.items():
    print(f"page {page_number:3}: {len(markdown):5} characters")

page   4:   795 characters
page   8:  1271 characters
page  13:  4099 characters
page  26:  5055 characters
page 128:  3520 characters
page 139:  2024 characters
page 142:  3957 characters


## 2. Chunk: split each page at its headings

Each page is split on its own, so every chunk has exactly one page number.

`MarkdownHeaderTextSplitter` cuts at every heading level and keeps each heading at the top of its section
(`strip_headers=False`). It ignores `#` lines inside code blocks, such as page 8's Mermaid diagram.
It also strips the indentation of every line; on these pages only the Mermaid lines are indented, and Mermaid ignores it.

`merge_small_sections` then joins a section shorter than `MIN_CHUNK_CHARACTERS` to the next one,
so a heading stays with what follows it. The last section of a page has no next one, so it joins the one before.

In [3]:
HEADINGS = [("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4"), ("#####", "h5"), ("######", "h6")]
heading_splitter = MarkdownHeaderTextSplitter(HEADINGS, strip_headers=False)  # keep each heading in its section


def split_into_sections(markdown):
    """Cut a page's markdown at its headings. Each section starts with its heading."""
    return [document.page_content for document in heading_splitter.split_text(markdown)]


def merge_small_sections(sections, min_characters):
    """Join each section shorter than min_characters to the next one.

    The last section of a page has no next one, so it joins the one before it.
    """
    chunks = []
    current = ""
    for section in sections:
        current = f"{current}\n\n{section}" if current else section
        if len(current) >= min_characters:
            chunks.append(current)
            current = ""
    if current and chunks:
        chunks[-1] = f"{chunks[-1]}\n\n{current}"
    elif current:
        chunks.append(current)
    return chunks


def build_text_chunks(markdown_by_page):
    """Split every page on its own, so each chunk belongs to exactly one page."""
    chunks = []
    for page_number, markdown in markdown_by_page.items():
        sections = split_into_sections(markdown)
        page_chunks = merge_small_sections(sections, config.MIN_CHUNK_CHARACTERS)
        for number, text in enumerate(page_chunks, start=1):
            chunks.append({"id": f"page{page_number}-{number}", "page_number": page_number, "text": text})
    return chunks


text_chunks = build_text_chunks(markdown_by_page)
print(f"{len(text_chunks)} text chunks from {len(markdown_by_page)} pages (notebook 05 made 11)\n")
for chunk in text_chunks:
    tables = chunk["text"].count("<table")
    print(f"{chunk['id']:<10} {len(chunk['text']):5} characters, {tables} table(s)")

13 text chunks from 7 pages (notebook 05 made 11)

page4-1      810 characters, 0 table(s)
page8-1     1157 characters, 0 table(s)
page13-1    1463 characters, 0 table(s)
page13-2    1476 characters, 0 table(s)
page13-3    1169 characters, 0 table(s)
page26-1    1683 characters, 1 table(s)
page26-2    1354 characters, 0 table(s)
page26-3    2022 characters, 1 table(s)
page128-1   1269 characters, 1 table(s)
page128-2    672 characters, 1 table(s)
page128-3   1582 characters, 1 table(s)
page139-1   2030 characters, 1 table(s)
page142-1   3961 characters, 1 table(s)


The chunks in full, to check where the pages were split:

In [4]:
for chunk in text_chunks:
    print(f"==================== {chunk['id']}")
    print(chunk["text"])
    print()

==================== page4-1
## **AI Is Driving the Largest Infrastructure Buildout in Human History**  
**Data centers are becoming AI factories.**  
Land, power, cooling, networking, and systems must scale in parallel.
Every company will use AI. Every country will build AI infrastructure.
This is the next industrial revolution. We are only a few hundred billion dollars into it. Trillions more still need to be built.  
image: infographic showing the evolution of DSX (Data Center Systems) from reference designs to liquid cooling  
**Reference Designs, Best Practices, Methodologies**  
**Rubin DSX Reference Designs**  
**DSX Ecosystem**  
**Omniverse DSX Blueprint**  
**DSX Max-Q**  
**DSX Flex**  
**DSX Exchange**  
**DSX Sim**  
**DSX Libraries, APIs, Software**  
**Chips, Systems, Facilities**  
**Power**  
**Liquid Cooling**

==================== page8-1
## **Agentic AI Enters Production**  
**AI now perceives, reasons, plans, and acts.**  
Agentic systems break complex problems int

## 3. Image documents

LlamaParse names each crop after its page and kind: `page_4_image_1_v2.jpg`, `page_8_chart_1_v2.jpg`,
`page_139_table_1_v2.jpg`. Images and charts are kept; page 8's agentic AI diagram is saved as a `chart`.
Table crops are skipped, because the table HTML is already in the text chunk.

LlamaParse sometimes crops parts of a diagram as well as the whole: on page 4, both rows of the DSX diagram also
came out as crops of their own. The parse result gives each crop's box on the page (`bbox`), so a figure crop with
at least 90% of its area inside a larger figure crop on the same page is skipped. The larger crop shows the whole
diagram, connections included.

In [5]:
def crop_kind(filename):
    """'image', 'chart' or 'table', read from a crop name such as page_8_chart_1_v2.jpg."""
    return filename.split("_")[2]


def crop_page(filename):
    """The page number, read from a crop name such as page_8_chart_1_v2.jpg."""
    return int(filename.split("_")[1])


def corners(crop):
    box = crop["bbox"]
    return box["x"], box["y"], box["x"] + box["w"], box["y"] + box["h"]


def share_inside(small, big):
    """How much of the small crop's area lies inside the big crop, from 0 to 1."""
    sx1, sy1, sx2, sy2 = corners(small)
    bx1, by1, bx2, by2 = corners(big)
    overlap_width = max(0, min(sx2, bx2) - max(sx1, bx1))
    overlap_height = max(0, min(sy2, by2) - max(sy1, by1))
    return overlap_width * overlap_height / ((sx2 - sx1) * (sy2 - sy1))


def is_nested(crop, page_crops):
    """True when a larger crop on the same page already holds at least 90% of this one."""
    area = crop["bbox"]["w"] * crop["bbox"]["h"]
    return any(
        other["bbox"]["w"] * other["bbox"]["h"] > area and share_inside(crop, other) >= 0.9
        for other in page_crops
    )


crops = [image for image in parse_result["images_content_metadata"]["images"] if image["category"] == "layout"]
crops = sorted(crops, key=lambda crop: (crop_page(crop["filename"]), crop["filename"]))
figure_crops = [crop for crop in crops if crop_kind(crop["filename"]) != "table"]
print(f"Crops by kind: {dict(Counter(crop_kind(crop['filename']) for crop in crops))}")

figures = []
for crop in figure_crops:
    page_number = crop_page(crop["filename"])
    page_crops = [other for other in figure_crops if crop_page(other["filename"]) == page_number]
    if is_nested(crop, page_crops):
        print(f"skipped {crop['filename']}: it lies inside a larger crop on page {page_number}")
        continue
    path = IMAGE_DIRECTORY / crop["filename"]
    figures.append({"id": path.stem, "page_number": page_number, "path": path})

print(f"{len(figures)} figures become image documents: {[figure['id'] for figure in figures]}")

Crops by kind: {'image': 3, 'chart': 1, 'table': 7}
skipped page_4_image_2_v2.jpg: it lies inside a larger crop on page 4
skipped page_4_image_3_v2.jpg: it lies inside a larger crop on page 4
2 figures become image documents: ['page_4_image_1_v2', 'page_8_chart_1_v2']


## 4. Describe each image

DeepSeek Flash gets the image and the text of its page, and writes a description of the image.
That description is what gets embedded. DeepSeek's API is OpenAI-compatible, so `ChatOpenAI` calls it
with `DESCRIPTION_BASE_URL`.

Flash is a reasoning model: its reasoning tokens count toward `DESCRIPTION_MAX_TOKENS` along with the description.
A description cut off by that limit is reported.

Descriptions are cached in `image_descriptions.json`, so running this notebook again does not call the model again.

In [6]:
IMAGE_PROMPT = """You are writing a searchable description of an image from page {page_number} of a company annual report.
The text of that page is below, for context. Describe the image itself, not the whole page.

PAGE TEXT:
{page_text}

Cover:
1. What kind of image it is (diagram, chart, photo, ...) and what it shows
2. Every label, name, number and connection you can read in it
3. What the image explains, in a few sentences
4. Questions it could answer, and other words people might search for

DESCRIPTION:"""


def description_model():
    """DeepSeek Flash through its OpenAI-compatible API. Rate limits and server errors are retried."""
    return ChatOpenAI(
        model=DESCRIPTION_MODEL,
        base_url=DESCRIPTION_BASE_URL,
        api_key=os.environ["DEEPSEEK_API_KEY"],
        temperature=0,
        max_tokens=config.DESCRIPTION_MAX_TOKENS,
        max_retries=config.MODEL_RETRY_ATTEMPTS,
    )


def describe_image(llm, image_path, page_number, page_text):
    """Ask the model for a searchable description of one image, with its page's text as context."""
    image = base64.b64encode(image_path.read_bytes()).decode()
    content = [
        {"type": "text", "text": IMAGE_PROMPT.format(page_number=page_number, page_text=page_text)},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image}"}},
    ]
    answer = llm.invoke([HumanMessage(content=content)])
    if answer.response_metadata.get("finish_reason") == "length":
        print(f"   {image_path.name}: description cut off at {config.DESCRIPTION_MAX_TOKENS} tokens")
    return answer.content


descriptions = json.loads(DESCRIPTIONS_PATH.read_text(encoding="utf-8")) if DESCRIPTIONS_PATH.exists() else {}
llm = description_model()

for figure in figures:
    if figure["id"] in descriptions:
        continue
    start = time.time()
    page_text = markdown_by_page[figure["page_number"]]
    descriptions[figure["id"]] = describe_image(llm, figure["path"], figure["page_number"], page_text)
    DESCRIPTIONS_PATH.write_text(json.dumps(descriptions, indent=1), encoding="utf-8")
    print(f"described {figure['id']} in {time.time() - start:.0f}s")

described = [figure["id"] for figure in figures if figure["id"] in descriptions]
print(f"{len(described)} of {len(figures)} figures have a description: {described}")

2 of 2 figures have a description: ['page_4_image_1_v2', 'page_8_chart_1_v2']


The descriptions in full:

In [7]:
for figure in figures:
    print(f"==================== {figure['id']}")
    print(descriptions.get(figure["id"], "(no description)"))
    print()

==================== page_4_image_1_v2
**1. Image type and overview**  
A wide, horizontal technical infographic/diagram on a black background. It maps the DSX data-center stack in two rows: a top row of reference designs, ecosystem, Omniverse blueprint, and DSX software/tools; a lower row showing power, a long row of server racks, and liquid cooling, connected by callout lines to a large 3D cutaway rendering of an AI data-center campus/factory at the bottom. The small thumbnail icons are illustrative; internal text in them is not legible.

**2. Readable labels, names, and connections**  
Readable labels, left to right and top to bottom:  
- **Reference Designs, Best Practices, Methodologies**  
- **Rubin DSX Reference Designs**  
- **DSX Ecosystem**  
- **Omniverse DSX Blueprint**  
- **DSX Max-Q**  
- **DSX Flex**  
- **DSX Exchange**  
- **DSX Sim**  
- **DSX Libraries, APIs, Software**  
- **Chips, Systems, Facilities**  
- **Power**  
- **Liquid Cooling**

Connections:  
- A horiz

## 5. Store in a test Chroma store

- **Text chunk:** the chunk text is embedded as it is, table HTML included. Metadata: `page_number`.
- **Image:** its description is embedded. Metadata: `page_number` and `image_path`, so the answering model can be given the image itself.

The test store lives in `notebooks/llamaparse_output/chroma_test_06`, apart from notebook 05's and the pipeline's.

In [8]:
def build_documents(text_chunks, figures, descriptions):
    """Text chunks are embedded as they are. Each figure is embedded by its description and keeps its image path."""
    documents = []
    for chunk in text_chunks:
        metadata = {"page_number": chunk["page_number"]}
        documents.append(Document(page_content=chunk["text"], metadata=metadata, id=chunk["id"]))
    for figure in figures:
        if figure["id"] not in descriptions:
            continue
        metadata = {"page_number": figure["page_number"], "image_path": figure["path"].relative_to(PROJECT_ROOT).as_posix()}
        documents.append(Document(page_content=descriptions[figure["id"]], metadata=metadata, id=figure["id"]))
    return documents


documents = build_documents(text_chunks, figures, descriptions)
test_store = Chroma(
    persist_directory=str(TEST_STORE_DIRECTORY),
    embedding_function=NemotronEmbeddings(),
    collection_metadata={"hnsw:space": "cosine"},
)
test_store.reset_collection()
test_store.add_documents(documents)
image_count = len(documents) - len(text_chunks)
print(f"Stored {len(text_chunks)} text chunks and {image_count} image documents in {TEST_STORE_DIRECTORY.relative_to(PROJECT_ROOT)}")

Stored 13 text chunks and 2 image documents in notebooks\llamaparse_output\chroma_test_06


## 6. Search

The questions from notebook 05, where the right page came first for 6 of 6. The last one is not answered
anywhere in these pages, so all its scores should be low, ideally under `MIN_RELEVANCE_SCORE`.

Under each question, "images for the answering model" lists the image documents that scored at least
`MIN_RELEVANCE_SCORE`. Those are the images the answering model would be given.

In [9]:
QUESTIONS = [
    ("What was NVIDIA's total revenue in fiscal year 2024?", 139),
    ("Which director nominees are not independent?", 26),
    ("What does NemoClaw connect to in the agentic AI architecture?", 8),
    ("What share of revenue went to research and development in fiscal 2026?", 128),
    ("How much did NVIDIA spend repurchasing shares in fiscal 2026?", 142),
    ("What are the DSX reference designs for AI factories?", 4),
    ("How many employees does NVIDIA have?", None),
]

first_place = 0
for question, expected_page in QUESTIONS:
    results = test_store.similarity_search_with_relevance_scores(question, k=config.RETRIEVED_CHUNK_COUNT)
    top_page = results[0][0].metadata["page_number"]
    verdict = "no answer expected" if expected_page is None else ("OK" if top_page == expected_page else "MISS")
    if top_page == expected_page:
        first_place += 1

    print(f"{question}\n   expected page {expected_page}, got page {top_page}: {verdict}")
    for document, score in results:
        kind = "image" if "image_path" in document.metadata else "text"
        preview = " ".join(document.page_content.split())[:80]
        print(f"   {score:.3f}  page {document.metadata['page_number']:3}  {kind:5}  {preview}")
    images = [document.metadata["image_path"] for document, score in results
              if "image_path" in document.metadata and score >= config.MIN_RELEVANCE_SCORE]
    print(f"   images for the answering model: {images or 'none'}")
    print()

answerable = sum(1 for _, page in QUESTIONS if page is not None)
print(f"Right page first for {first_place} of {answerable} answerable questions (notebook 05: 6 of 6)")

What was NVIDIA's total revenue in fiscal year 2024?
   expected page 139, got page 139: OK
   0.544  page 139  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Inco
   0.422  page 142  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Shar
   0.351  page  13  text   ## **Driving the AI Platform Shift** GTC is where the industry comes together ac
   images for the answering model: none



Which director nominees are not independent?
   expected page 26, got page 26: OK
   0.522  page  26  text   ## **Election of Directors (Proposal 1)** The following table provides summary i
   0.399  page  26  text   #### *Nominee Skills, Competencies, and Attributes* Below are the skills, compet
   0.356  page  26  text   ### **Recent Refreshment and Nominee Qualifications** Our director nominees refl
   images for the answering model: none



What does NemoClaw connect to in the agentic AI architecture?
   expected page 8, got page 8: OK
   0.626  page   8  image  **Searchable image description** **1. Image type / subject** A hand-drawn-style 
   0.521  page   8  text   ## **Agentic AI Enters Production** **AI now perceives, reasons, plans, and acts
   0.237  page  13  text   Compute is no longer just a cost to support software; it is a productive asset t
   images for the answering model: ['notebooks/llamaparse_output/images/page_8_chart_1_v2.jpg']



What share of revenue went to research and development in fiscal 2026?
   expected page 128, got page 128: OK
   0.483  page 128  text   The following table sets forth, for the periods indicated, certain items in our 
   0.339  page 139  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Inco
   0.326  page 128  text   ## **Reportable Segments** ### **Revenue by Reportable Segments** <table><tr><th
   images for the answering model: none



How much did NVIDIA spend repurchasing shares in fiscal 2026?
   expected page 142, got page 142: OK
   0.499  page 142  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Shar
   0.475  page 139  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Inco
   0.294  page 128  text   The following table sets forth, for the periods indicated, certain items in our 
   images for the answering model: none



What are the DSX reference designs for AI factories?
   expected page 4, got page 4: OK
   0.574  page   4  text   ## **AI Is Driving the Largest Infrastructure Buildout in Human History** **Data
   0.506  page   4  image  **1. Image type and overview** A wide, horizontal technical infographic/diagram 
   0.305  page  13  text   Compute is no longer just a cost to support software; it is a productive asset t
   images for the answering model: ['notebooks/llamaparse_output/images/page_4_image_1_v2.jpg']



How many employees does NVIDIA have?
   expected page None, got page 139: no answer expected
   0.375  page 139  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Inco
   0.352  page 142  text   ## **NVIDIA Corporation and Subsidiaries** ### **Consolidated Statements of Shar
   0.264  page  13  text   Compute is no longer just a cost to support software; it is a productive asset t
   images for the answering model: none

Right page first for 6 of 6 answerable questions (notebook 05: 6 of 6)


## Findings

**The approach works on these pages.** The right page came first for 6 of 6 answerable questions, as in notebook 05,
with tables embedded as raw HTML and only the images described. This run used a fresh parse of the seven pages:
the `agentic` tier returns slightly different markdown each time, and page 4 got three crops here against two before.

- **Chunking.** The 7 pages became 13 chunks (notebook 05: 11).
  - No heading was left alone: the `## NVIDIA Corporation and Subsidiaries` headings on pages 139 and 142 stay with their tables.
  - No table was cut, and page 8's Mermaid block stayed whole in one chunk.
  - Pages 13 and 128 start with a chunk that has no heading (text continued from the page before). Both still ranked first for their questions.
- **Tables without descriptions.** The table pages ranked first with scores close to notebook 05's, where tables were embedded by a description:

  | Page | This notebook | Notebook 05 |
  |---|---|---|
  | 26 | 0.522 | 0.500 |
  | 128 | 0.483 | 0.460 |
  | 139 | 0.544 | 0.587 |
  | 142 | 0.499 | 0.556 |

  Page 142 is the closest call: 0.499 against 0.475 for page 139, much as in notebook 05 (0.556 against 0.536).
- **Nested crops, skipped.** LlamaParse cropped page 4's diagram three times: the whole diagram (`page_4_image_1`) and its top-left and top-right rows, whose boxes lie 98.7% and 100% inside the first. Skipping a crop that lies at least 90% inside a larger one leaves one image document per diagram: 2 for these pages instead of 4.
- **Image documents.** Each diagram question sends exactly one image to the answering model.
  - NemoClaw question: page 8's image document ranked first (0.626) and its page text, with the Mermaid version, second, so the answering model gets both.
  - DSX question: the page text first (0.574), then page 4's image document (0.506).
  - Questions about text and tables brought no images with them.
- **Descriptions (DeepSeek Flash).** 19 seconds for page 4's diagram and 44 for page 8's; neither was cut off by `DESCRIPTION_MAX_TOKENS`.
  - Page 8: every label, and every connection but one (it misses the arrow between Sub-agents and Skills).
  - Page 4: all 12 labels. It reads the top row as one connected line, where the picture shows two groups.

**To watch on the full report:**

- **The cutoff.** The question with no answer in these pages still scored 0.375 at the top, above the 0.3 `MIN_RELEVANCE_SCORE`, as in notebook 05. Re-check it on the full report.
- **Photos.** These pages had none. Count the image crops on the full report before describing them all.
